In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

#### Working on the development set

In [ ]:
data_path = Path("../data/development_set/dataset_labels.csv")

In [ ]:
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
def dataset_stats(data, name):
    total = len(data)
    positives = (data["label"] == 1).sum()
    negatives = (data["label"] == 0).sum()

    uncertain = data["uncertain"].notna().sum()
    certain = data["uncertain"].isna().sum()

    certain_data = data[data["uncertain"].isna()]
    certain_positives = (certain_data["label"] == 1).sum()
    certain_negatives = (certain_data["label"] == 0).sum()

    return {
        "Dataset": name,
        "N": total,
        
        "Positive": positives,
        "Negative": negatives,
        "Uncertain": uncertain,
        "Certain": certain,
        "N after removing uncertain": len(certain_data),
        "Positive after removing uncertain": certain_positives,
        "Negative after removing uncertain": certain_negatives,
    }


stats = [
    dataset_stats(df, "Full"),
    dataset_stats(df[df["split"] == "train"], "Train"),
    dataset_stats(df[df["split"] == "test"], "Test"),
]

dataset_stats_table = pd.DataFrame(stats).set_index("Dataset")

dataset_stats_table

In [ ]:
qwen_label_columns = sorted(
    column for column in df.columns if column.startswith("qwen_s")
)
if not qwen_label_columns:
    raise ValueError("No model-label columns matching 'qwen_s*' were found")

uncertainty_column = next(
    (
        column
        for column in ("uncertainty", "uncertain")
        if column in df.columns
    ),
    None,
)
if uncertainty_column is None:
    raise ValueError(
        "No uncertainty column named 'uncertainty' or 'uncertain' was found"
    )

uncertainty_values = pd.to_numeric(
    df[uncertainty_column], errors="coerce"
)
scope_masks = {
    "all": pd.Series(True, index=df.index),
    # Missing/blank uncertainty is included because it is not equal to 1.
    "certain": uncertainty_values.ne(1),
}

metric_rows = []
for model_column in qwen_label_columns:
    for scope_name, scope_mask in scope_masks.items():
        for split_name in ("train", "test"):
            split_rows = df.loc[
                df["split"].eq(split_name) & scope_mask,
                ["label", model_column],
            ].apply(pd.to_numeric, errors="coerce").dropna()

            if split_rows.empty:
                raise ValueError(
                    f"No evaluable rows for {model_column!r} in "
                    f"{split_name!r}, scope={scope_name!r}"
                )

            y_true = split_rows["label"].astype(int)
            y_pred = split_rows[model_column].astype(int)
            observed_labels = set(y_true) | set(y_pred)
            if not observed_labels <= {0, 1}:
                raise ValueError(
                    f"Non-binary values for {model_column!r} in "
                    f"{split_name!r}: {sorted(observed_labels)}"
                )

            metric_rows.append(
                {
                    "qwen_label": model_column,
                    "scope": scope_name,
                    "split": split_name,
                    "samples": len(split_rows),
                    "accuracy": accuracy_score(y_true, y_pred),
                    "precision": precision_score(
                        y_true, y_pred, pos_label=1, zero_division=0
                    ),
                    "recall": recall_score(
                        y_true, y_pred, pos_label=1, zero_division=0
                    ),
                    "f1": f1_score(
                        y_true, y_pred, pos_label=1, zero_division=0
                    ),
                }
            )

qwen_metrics = pd.DataFrame(metric_rows)


In [ ]:
qwen_metrics.style.format(
    {
        "accuracy": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}",
    }
).hide(axis="index")

In [ ]:
metric_order = ["accuracy", "precision", "recall", "f1"]
split_order = ["train", "test"]

def grouped_metric_table(scope_name):
    table = (
        qwen_metrics.loc[qwen_metrics["scope"].eq(scope_name)]
        .pivot(
            index="qwen_label",
            columns="split",
            values=metric_order,
        )
        .swaplevel(0, 1, axis=1)
        .reindex(
            columns=pd.MultiIndex.from_product(
                [split_order, metric_order],
                names=["dataset", "metric"],
            )
        )
    )
    table.index.name = "model"
    return table.rename(
        columns={
            "train": "Train",
            "test": "Test",
            "accuracy": "Accuracy",
            "precision": "Precision",
            "recall": "Recall",
            "f1": "F1",
        }
    )

qwen_metrics_table = grouped_metric_table("all")
qwen_metrics_table.style.format("{:.3f}").set_caption(
    "All datapoints"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

In [ ]:
qwen_certain_metrics_table = grouped_metric_table("certain")
certain_counts = {
    split_name: int(
        (df["split"].eq(split_name) & scope_masks["certain"]).sum()
    )
    for split_name in split_order
}
qwen_certain_metrics_table.style.format("{:.3f}").set_caption(
    f"Certain datapoints only ({uncertainty_column} != 1): "
    f"Train n={certain_counts['train']}, "
    f"Test n={certain_counts['test']}"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

In [ ]:
data_path = Path("../data/development_set/dataset_labels_category.csv")

In [ ]:
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
repetition_columns = {}
for column in df.columns:
    if "_K" not in column or "catcol" in column.lower():
        continue
    model_name, repetition_number = column.rsplit("_K", 1)
    if repetition_number in {"1", "2", "3"}:
        repetition_columns.setdefault(model_name, {})[
            f"K{repetition_number}"
        ] = column

if not repetition_columns:
    raise ValueError("No repeated model columns ending in _K1, _K2, or _K3 were found")

expected_repetitions = {"K1", "K2", "K3"}
for model_name, columns_by_repetition in repetition_columns.items():
    missing_repetitions = expected_repetitions - set(columns_by_repetition)
    if missing_repetitions:
        raise ValueError(
            f"{model_name!r} is missing repetitions: "
            f"{sorted(missing_repetitions)}"
        )

uncertainty_column = next(
    (column for column in ("uncertainty", "uncertain") if column in df.columns),
    None,
)
if uncertainty_column is None:
    raise ValueError(
        "No uncertainty column named 'uncertainty' or 'uncertain' was found"
    )

uncertainty_values = pd.to_numeric(df[uncertainty_column], errors="coerce")
category_scope_masks = {
    "all": pd.Series(True, index=df.index),
    "certain": uncertainty_values.ne(1),
}
category_metric_order = ["accuracy", "precision", "recall", "f1"]
category_split_order = ["train", "test"]

repetition_metric_rows = []
for model_name, columns_by_repetition in sorted(repetition_columns.items()):
    for repetition in ("K1", "K2", "K3"):
        prediction_column = columns_by_repetition[repetition]
        for scope_name, scope_mask in category_scope_masks.items():
            for split_name in category_split_order:
                split_rows = df.loc[
                    df["split"].eq(split_name) & scope_mask,
                    ["label", prediction_column],
                ].apply(pd.to_numeric, errors="coerce").dropna()

                if split_rows.empty:
                    raise ValueError(
                        f"No evaluable rows for {prediction_column!r} in "
                        f"{split_name!r}, scope={scope_name!r}"
                    )

                y_true = split_rows["label"].astype(int)
                y_pred = split_rows[prediction_column].astype(int)
                observed_labels = set(y_true) | set(y_pred)
                if not observed_labels <= {0, 1}:
                    raise ValueError(
                        f"Non-binary values for {prediction_column!r}: "
                        f"{sorted(observed_labels)}"
                    )

                repetition_metric_rows.append(
                    {
                        "model": model_name,
                        "repetition": repetition,
                        "scope": scope_name,
                        "split": split_name,
                        "samples": len(split_rows),
                        "accuracy": accuracy_score(y_true, y_pred),
                        "precision": precision_score(
                            y_true, y_pred, pos_label=1, zero_division=0
                        ),
                        "recall": recall_score(
                            y_true, y_pred, pos_label=1, zero_division=0
                        ),
                        "f1": f1_score(
                            y_true, y_pred, pos_label=1, zero_division=0
                        ),
                    }
                )

category_repetition_metrics = pd.DataFrame(repetition_metric_rows)

summary_rows = []
for (model_name, scope_name, split_name), group in category_repetition_metrics.groupby(
    ["model", "scope", "split"], sort=True
):
    summary_row = {
        "model": model_name,
        "scope": scope_name,
        "split": split_name,
        "samples": int(group["samples"].iloc[0]),
        "repetitions": len(group),
    }
    for metric in category_metric_order:
        summary_row[f"{metric}_mean"] = group[metric].mean()
        summary_row[f"{metric}_std"] = group[metric].std(ddof=1)
    summary_rows.append(summary_row)

category_metric_summary = pd.DataFrame(summary_rows)

In [ ]:
category_repetition_metrics.style.format(
    {metric: "{:.3f}" for metric in category_metric_order}
).hide(axis="index").set_caption(
    "Metrics for each repetition (K1, K2, K3)"
)

In [ ]:
def grouped_repetition_metric_table(scope_name):
    scope_summary = category_metric_summary.loc[
        category_metric_summary["scope"].eq(scope_name)
    ]
    model_order = sorted(scope_summary["model"].unique())
    table = pd.DataFrame(index=pd.Index(model_order, name="model"))

    for split_name in category_split_order:
        split_summary = scope_summary.loc[
            scope_summary["split"].eq(split_name)
        ].set_index("model")
        for metric in category_metric_order:
            means = split_summary[f"{metric}_mean"].reindex(model_order)
            standard_deviations = split_summary[f"{metric}_std"].reindex(
                model_order
            )
            table[(split_name.title(), metric.title())] = [
                f"{mean:.3f} ± {std:.3f}"
                for mean, std in zip(means, standard_deviations)
            ]

    table.columns = pd.MultiIndex.from_tuples(
        table.columns, names=["dataset", "metric"]
    )
    return table

category_metrics_table = grouped_repetition_metric_table("all")
category_metrics_table.style.set_caption(
    "All datapoints — mean ± sample SD across K1, K2, and K3"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

In [ ]:
category_certain_metrics_table = grouped_repetition_metric_table("certain")
category_certain_counts = {
    split_name: int(
        (df["split"].eq(split_name) & category_scope_masks["certain"]).sum()
    )
    for split_name in category_split_order
}
category_certain_metrics_table.style.set_caption(
    f"Certain datapoints only ({uncertainty_column} != 1) — "
    f"mean ± sample SD across K1, K2, and K3; "
    f"Train n={category_certain_counts['train']}, "
    f"Test n={category_certain_counts['test']}"
).set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [("text-align", "center")],
        }
    ]
)

#### Creating manifest unid files for classification handling

In [3]:
from contextlib import closing
import math
import sqlite3

central_db_path = Path("../data/central_papers.db").resolve()
uid_manifest_dir = Path("../data/uid_manifest").resolve()
uids_per_file = 10_000
manifest_prefix = "classify_uids_"

if not central_db_path.is_file():
    raise FileNotFoundError(f"Central database not found: {central_db_path}")
uid_manifest_dir.mkdir(parents=True, exist_ok=True)

# Remove temporary files left by an interrupted earlier generation.
for temporary_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt.tmp"):
    temporary_path.unlink()

database_uri = f"{central_db_path.as_uri()}?mode=ro"
manifest_rows = []
staged_files = []
written_uid_count = 0

with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")

    table_exists = conn.execute(
        """
        SELECT 1
        FROM sqlite_master
        WHERE type = 'table' AND name = 'classify'
        """
    ).fetchone()
    if table_exists is None:
        raise ValueError("central_papers.db has no classify table")

    available_columns = {
        row[1] for row in conn.execute("PRAGMA table_info(classify)")
    }
    required_columns = {"paper_id", "paper_uid"}
    missing_columns = sorted(required_columns - available_columns)
    if missing_columns:
        raise ValueError(
            "classify is missing column(s): " + ", ".join(missing_columns)
        )

    total_uid_count, unique_uid_count = conn.execute(
        "SELECT COUNT(*), COUNT(DISTINCT paper_uid) FROM classify"
    ).fetchone()
    if total_uid_count != unique_uid_count:
        raise ValueError(
            f"classify contains {total_uid_count - unique_uid_count:,} "
            "duplicate paper_uid value(s)"
        )

    cursor = conn.execute(
        "SELECT paper_uid FROM classify ORDER BY paper_id"
    )
    file_number = 0
    while True:
        rows = cursor.fetchmany(uids_per_file)
        if not rows:
            break

        uids = [str(row[0]).strip() for row in rows]
        if any(not uid for uid in uids):
            raise ValueError("classify contains an empty paper_uid")

        file_number += 1
        final_path = uid_manifest_dir / (
            f"{manifest_prefix}{file_number:04d}.txt"
        )
        temporary_path = final_path.with_suffix(".txt.tmp")
        temporary_path.write_text(
            "\n".join(uids) + "\n",
            encoding="utf-8",
        )
        staged_files.append((temporary_path, final_path))
        written_uid_count += len(uids)
        manifest_rows.append(
            {
                "file": final_path.name,
                "uids": len(uids),
                "first_uid": uids[0],
                "last_uid": uids[-1],
            }
        )

expected_file_count = math.ceil(total_uid_count / uids_per_file)
if written_uid_count != total_uid_count:
    raise RuntimeError(
        f"Expected {total_uid_count:,} UIDs, wrote {written_uid_count:,}"
    )
if len(staged_files) != expected_file_count:
    raise RuntimeError(
        f"Expected {expected_file_count:,} files, staged {len(staged_files):,}"
    )
if any(row["uids"] > uids_per_file for row in manifest_rows):
    raise RuntimeError("A manifest exceeds the 10,000-UID limit")

# Replace only manifests created by this cell; leave unrelated files alone.
for old_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt"):
    old_path.unlink()
for temporary_path, final_path in staged_files:
    temporary_path.replace(final_path)

uid_manifest_summary = pd.DataFrame(manifest_rows)
print(f"Manifest directory: {uid_manifest_dir}")
print(f"UIDs written: {written_uid_count:,}")
print(f"Manifest files: {len(staged_files):,}")
print(f"Maximum UIDs per file: {uids_per_file:,}")
uid_manifest_summary

Manifest directory: /Users/kevinge/Work/Data Extraction/synth_extract/data/uid_manifest
UIDs written: 1,012,267
Manifest files: 102
Maximum UIDs per file: 10,000


,file,uids,first_uid,last_uid
0,classify_uids_0001.txt,10000,ID000000002,ID000011897
1,classify_uids_0002.txt,10000,ID000011898,ID000029643
2,classify_uids_0003.txt,10000,ID000029644,ID000039661
3,classify_uids_0004.txt,10000,ID000039662,ID000049831
4,classify_uids_0005.txt,10000,ID000049832,ID000059926
...,...,...,...,...
97,classify_uids_0098.txt,10000,ID001055120,ID001066875
98,classify_uids_0099.txt,10000,ID001066876,ID001078146
99,classify_uids_0100.txt,10000,ID001078147,ID001088423
100,classify_uids_0101.txt,10000,ID001088424,ID001099501
